# 🧑‍💻 파라미터 실험 — 자율 실습

---

## 📌 실습 방법

- `____` 부분을 직접 채워 실험을 완성하세요
- 막히면 💡 **힌트** 를 먼저 확인하세요
- 그래도 모르면 노트북 맨 아래 **정답 코드 셀** 을 확인하세요

```
MISSION 1. 나만의 주제로 temperature 4단계 실험
MISSION 2. 반복 실험으로 동일성 확인
MISSION 3. finish_reason: length 직접 만들기
MISSION 4. top_p 실험 + temperature와 조합
MISSION 5. 내 도메인에 맞는 파라미터 조합 도출
MISSION 6. 실험 전체 비용 계산
```

> ⏱️ 예상 시간: **60~80분**

---
## 🔧 사전 준비 — 실행만 하세요

In [ ]:
# 라이브러리 + 클라이언트 + 비용 추적기 (수정 금지)
from google.colab import userdata
import openai

client = openai.OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

class TokenTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    def add(self, response):
        self.total_input  += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count   += 1

    def report(self):
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰")
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")

tracker = TokenTracker()
print("✅ 준비 완료!")

---
## MISSION 1. 나만의 주제로 temperature 4단계 실험

강의에서는 법률 질문을 썼습니다. 이번엔 **내가 관심 있는 주제**로 직접 실험해보세요.

**조건**
- `my_question` 에 내가 원하는 질문을 넣으세요 (사실형 질문 권장)
- temperature를 `[0.0, 0.5, 1.0, 1.5]` 순서로 바꿔가며 호출하세요
- 각 응답과 `finish_reason`, `completion_tokens` 를 출력하세요
- 마지막에 `tracker.add(resp)` 로 비용을 기록하세요

> 💡 **힌트:** 강의 실습의 `question` 변수를 내 질문으로 바꾸고,  
> `for temp in [0.0, 0.5, 1.0, 1.5]:` 반복문 안에서 `temperature=temp` 로 설정합니다

In [ ]:
# 내가 관심 있는 주제의 질문을 넣으세요
my_question = "____"   # 예: "파이썬을 배우면 어떤 일을 할 수 있나요?"

for temp in [0.0, 0.5, 1.0, 1.5]:   # temperature 4단계
    print(f"\n{'='*50}")
    print(f"[temperature = {temp}]")
    print(f"{'='*50}")

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=____,   # 반복 변수를 넣으세요
        max_tokens=100,
        top_p=0.9,
    )

    print(____)
    print(f"\n  → 종료 이유: {____}")
    print(f"  → 출력 토큰: {____}")

    tracker.add(resp)

**📝 MISSION 1 기록표 — 직접 채우세요**

| temperature | 응답이 비슷한가? | 어휘가 다양한가? | 특이사항 |
|------------|----------------|----------------|----------|
| `0.0` | | | |
| `0.5` | | | |
| `1.0` | | | |
| `1.5` | | | |

---
## MISSION 2. 반복 실험으로 동일성 확인

MISSION 1에서 쓴 `my_question` 으로 temperature=0.0 실험을 **3번 반복**하세요.  
3회 응답이 모두 같은지 자동으로 확인하는 코드도 완성하세요.

**조건**
- `range(3)` 으로 3번 반복하세요
- 각 응답을 `responses` 리스트에 저장하세요
- 3회 응답이 같은지 `if` 로 확인하고 결과를 출력하세요

> 💡 **힌트:** `responses.append(answer)` 로 리스트에 추가하고,  
> `responses[0] == responses[1] == responses[2]` 로 동일성을 확인합니다

In [ ]:
print("─── temperature=0.0 반복 실험 (3회) ───")

responses = []   # 응답을 저장할 빈 리스트

for i in range(____):   # 3번 반복하세요
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=____,   # 0.0으로 고정하세요
        max_tokens=100,
    )
    answer = resp.choices[0].message.content
    responses.______(answer)   # 리스트에 추가하세요
    print(f"\n[{i+1}회차]")
    print(answer[:100])
    tracker.add(resp)

# 3회 응답이 모두 같은지 확인하세요
if ____ == ____ == ____:
    print("\n✅ 3회 모두 동일! (temperature=0.0은 결정론적)")
else:
    print("\n⚠️ 약간의 차이 발생")

---
## MISSION 3. finish_reason: length 직접 만들기

max_tokens를 **의도적으로 작게** 설정해서 문장이 잘리는 현상을 만들어보세요.  
그 다음 값을 늘려서 정상 종료(`stop`)가 되도록 고쳐보세요.

**조건**
- 첫 번째 호출: `max_tokens=20` (잘림 발생 예상)
- 두 번째 호출: `max_tokens=____` (정상 종료되도록 직접 값 정하기)
- 두 번 모두 `finish_reason` 을 출력해서 비교하세요

> 💡 **힌트:** `finish_reason` 이 `"length"` 면 잘린 것, `"stop"` 이면 정상 종료입니다

In [ ]:
q_long = "파이썬의 장점을 5가지 자세하게 설명해 주세요."

# ① max_tokens=20 (의도적으로 작게)
resp_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": q_long}],
    temperature=0.3,
    max_tokens=____,   # 20으로 설정하세요
)

print("[max_tokens=20]")
print(resp_short.choices[0].message.content)
print("종료 이유:", ____)   # finish_reason 을 출력하세요
tracker.add(resp_short)

print("\n" + "─"*50)

# ② max_tokens를 직접 정해서 stop이 나오도록
resp_full = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": q_long}],
    temperature=0.3,
    max_tokens=____,   # stop이 나오도록 값을 정하세요
)

print(f"[max_tokens={____}]")
print(resp_full.choices[0].message.content)
print("종료 이유:", ____)   # finish_reason 을 출력하세요
tracker.add(resp_full)

---
## MISSION 4. top_p 실험 + temperature와 조합

top_p를 바꿔가며 실험하고, temperature·top_p 조합을 직접 만들어보세요.

**조건 (파트 A)**
- `top_p` 를 `[0.3, 0.9, 1.0]` 순서로 바꿔가며 호출하세요
- `temperature=0.7` 로 고정하세요

**조건 (파트 B)**
- 내가 생각하는 "법률 Q&A 최적 조합" 을 직접 설정해 호출하세요
- temperature와 top_p 값을 자유롭게 정하세요

> 💡 **힌트:** 파트 A는 `for p in [0.3, 0.9, 1.0]:` 로 반복, `top_p=p` 로 설정합니다

In [ ]:
# ─── 파트 A: top_p 3단계 실험 ───
for p in [____]:   # [0.3, 0.9, 1.0] 을 채우세요
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=____,   # 0.7로 고정하세요
        max_tokens=100,
        top_p=____,         # 반복 변수를 넣으세요
    )
    print(f"\n[top_p={p}]")
    print(resp.choices[0].message.content[:120])
    tracker.add(resp)

print("\n" + "="*50)
print("─── 파트 B: 내가 생각하는 법률 Q&A 최적 조합 ───")

# ─── 파트 B: 내가 직접 조합 설정 ───
resp_best = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "계약 해지 시 손해배상 청구 요건은 무엇인가요?"}],
    temperature=____,   # 내가 생각하는 최적값을 설정하세요
    max_tokens=200,
    top_p=____,         # 내가 생각하는 최적값을 설정하세요
)

print(f"내 설정 → temperature: ??  top_p: ??")
print(resp_best.choices[0].message.content)
tracker.add(resp_best)

---
## MISSION 5. 내 도메인에 맞는 파라미터 조합 도출

지금까지의 실험을 바탕으로, **내가 만들고 싶은 AI 서비스**에 맞는 파라미터 조합을 직접 설정하고 테스트해보세요.

**조건**
- 어떤 서비스인지 `service_name` 에 쓰세요
- 그 서비스에 맞는 `system` 프롬프트를 설정하세요
- temperature, max_tokens, top_p 를 직접 정하고 이유를 주석으로 쓰세요
- 테스트 질문 3개를 보내고 결과를 확인하세요

> 💡 **힌트:** 정확성이 중요한 서비스 → temperature 낮게 / 창의성이 중요 → temperature 높게

In [ ]:
# 내가 만들고 싶은 AI 서비스
service_name = "____"   # 예: "의료 Q&A 챗봇", "여행 추천 AI", "코딩 도우미"

system_prompt = "____"  # 그 서비스에 맞는 system 프롬프트를 쓰세요

# 내가 선택한 파라미터 (이유를 주석으로 쓰세요)
my_temperature = ____   # 이유: ____
my_max_tokens  = ____   # 이유: ____
my_top_p       = ____   # 이유: ____

# 테스트 질문 3개
test_questions = [
    "____",   # 첫 번째 테스트 질문
    "____",   # 두 번째 테스트 질문
    "____",   # 세 번째 테스트 질문
]

print(f"🤖 서비스: {service_name}")
print(f"   temperature={my_temperature} / max_tokens={my_max_tokens} / top_p={my_top_p}\n")

for q in test_questions:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": q},
        ],
        temperature=my_temperature,
        max_tokens=my_max_tokens,
        top_p=my_top_p,
    )
    print(f"Q: {q}")
    print(f"A: {resp.choices[0].message.content}\n")
    tracker.add(resp)

---
## MISSION 6. 전체 실험 비용 계산

MISSION 1~5에서 한 모든 호출의 **누적 비용**을 출력하세요.  
`tracker.report()` 한 줄이면 됩니다.

> 💡 **힌트:** 각 미션에서 `tracker.add(resp)` 를 빠뜨리지 않았다면 자동으로 합산됩니다

In [ ]:
# MISSION 1~5 전체 누적 비용을 출력하세요
print("💰 전체 실험 누적 비용")
____   # tracker.report() 를 호출하세요

---
## 🌟 도전 과제

### 도전 1. 환각 재현하기
- temperature=1.5 + 내가 잘 아는 분야(예: 파이썬 공식 문서, 특정 역사적 사실)로 질문
- GPT가 사실이 아닌 내용을 자신 있게 말하는지 확인하세요

### 도전 2. 파라미터 실험 자동 비교표 만들기
- temperature를 `[0.0, 0.3, 0.7, 1.0, 1.5]` 5단계로 바꾸며
- 각 응답의 **글자 수**를 자동으로 출력하는 코드를 만들어보세요
- `len(응답 텍스트)` 를 활용하세요

### 도전 3. 같은 질문 5회 반복 → 응답 다양성 점수 계산
- temperature=1.0에서 같은 질문을 5회 호출
- 5개 응답 중 중복이 몇 개인지 `set()` 으로 확인하세요
- `len(set(responses))` 가 높을수록 다양성이 높습니다

In [ ]:
# 도전 과제를 여기에 자유롭게 작성하세요


---
## 📂 정답 코드

> ⚠️ 꼭 혼자 먼저 시도한 뒤에 확인하세요!

In [ ]:
# ✅ MISSION 1 정답 ─ temperature 4단계 실험
my_question = "파이썬을 배우면 어떤 일을 할 수 있나요?"   # 예시 질문

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n{'='*50}")
    print(f"[temperature = {temp}]")
    print(f"{'='*50}")

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=temp,   # 반복 변수 그대로
        max_tokens=100,
        top_p=0.9,
    )

    print(resp.choices[0].message.content)                      # 답변 출력
    print(f"\n  → 종료 이유: {resp.choices[0].finish_reason}")  # finish_reason
    print(f"  → 출력 토큰: {resp.usage.completion_tokens}")     # 토큰 수
    tracker.add(resp)

In [ ]:
# ✅ MISSION 2 정답 ─ 반복 3회 동일성 확인
print("─── temperature=0.0 반복 실험 (3회) ───")

responses = []   # 빈 리스트

for i in range(3):   # 3번 반복
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=0.0,   # 고정
        max_tokens=100,
    )
    answer = resp.choices[0].message.content
    responses.append(answer)   # 리스트에 추가
    print(f"\n[{i+1}회차]")
    print(answer[:100])
    tracker.add(resp)

# 동일성 확인
if responses[0] == responses[1] == responses[2]:
    print("\n✅ 3회 모두 동일!")
else:
    print("\n⚠️ 약간의 차이 발생")

In [ ]:
# ✅ MISSION 3 정답 ─ finish_reason: length 만들기
q_long = "파이썬의 장점을 5가지 자세하게 설명해 주세요."

# ① 작은 값 → 잘림
resp_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": q_long}],
    temperature=0.3,
    max_tokens=20,   # 의도적으로 작게
)
print("[max_tokens=20]")
print(resp_short.choices[0].message.content)
print("종료 이유:", resp_short.choices[0].finish_reason)   # → length
tracker.add(resp_short)

print("\n" + "─"*50)

# ② 충분히 큰 값 → 정상 종료
resp_full = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": q_long}],
    temperature=0.3,
    max_tokens=400,   # 충분히 크게
)
print("[max_tokens=400]")
print(resp_full.choices[0].message.content)
print("종료 이유:", resp_full.choices[0].finish_reason)   # → stop
tracker.add(resp_full)

In [ ]:
# ✅ MISSION 4 정답 ─ top_p 실험 + 조합

# 파트 A: top_p 3단계
for p in [0.3, 0.9, 1.0]:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": my_question}],
        temperature=0.7,   # 고정
        max_tokens=100,
        top_p=p,           # 반복 변수
    )
    print(f"\n[top_p={p}]")
    print(resp.choices[0].message.content[:120])
    tracker.add(resp)

# 파트 B: 법률 Q&A 최적 조합 (예시)
resp_best = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "계약 해지 시 손해배상 청구 요건은?"}],
    temperature=0.2,   # 정확성 우선 → 낮게
    max_tokens=200,
    top_p=0.9,         # 실무 권장 고정값
)
print("\n[법률 Q&A 최적 조합: temperature=0.2, top_p=0.9]")
print(resp_best.choices[0].message.content)
tracker.add(resp_best)

In [ ]:
# ✅ MISSION 5 정답 예시 ─ 의료 Q&A 챗봇
service_name   = "의료 Q&A 챗봇"
system_prompt  = "당신은 의료 정보를 안내하는 AI입니다. 정확한 정보를 간결하게 답변하세요."
my_temperature = 0.2   # 정확성 최우선 → 낮게
my_max_tokens  = 200   # 충분하지만 너무 길지 않게
my_top_p       = 0.9   # 실무 권장 고정값

test_questions = ["두통의 원인은 무엇인가요?", "감기와 독감의 차이는?", "충분한 수면 시간은 몇 시간인가요?"]

print(f"🤖 서비스: {service_name}")
for q in test_questions:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": q}],
        temperature=my_temperature,
        max_tokens=my_max_tokens,
        top_p=my_top_p,
    )
    print(f"Q: {q}")
    print(f"A: {resp.choices[0].message.content}\n")
    tracker.add(resp)

In [ ]:
# ✅ MISSION 6 정답 ─ 전체 비용 출력
print("💰 전체 실험 누적 비용")
tracker.report()   # 한 줄이면 됩니다